# NeuroKit2 EOG processing and right-edge labels

This portable notebook covers only the EOG procedure used in the online EEG pipeline. Keep `run_002_train.npz` in the same directory as this notebook. It works from an arbitrary local folder and from Google Drive/Colab:

1. load EOG hardware channel 5 from Run 002;
2. robustly normalize and differentiate it;
3. low-pass filter the derivative at 10 Hz;
4. use NeuroKit2 to clean the signal, detect peaks, and find event boundaries;
5. merge overlapping/nearby detections; and
6. change the alternating sample label at the **rightmost end of every merged EOG event**.

The plots also show all four recorded EEG channels for temporal context, matching the original notebooks. Those EEG traces are loaded unchanged and do not participate in EOG detection or labeling. There is no EEG preprocessing, audio-cue matching, model training, or ocular-artifact subtraction in this tutorial.

## 1. Install the libraries

Run this once in the active Jupyter kernel. Restart the kernel if requested.

In [ ]:
%pip install numpy pandas scipy matplotlib neurokit2 ipynbname


In [ ]:
# All imports are centralized in this cell.
from pathlib import Path

import matplotlib.pyplot as plt
import neurokit2 as nk
import numpy as np
import pandas as pd
from scipy.signal import butter, sosfilt, sosfiltfilt

try:
    import ipynbname
except ImportError:
    ipynbname = None

try:
    from google.colab import drive
except ImportError:
    drive = None


## 2. Mount Google Drive (Colab only)

In Google Colab, authorize this cell to mount `MyDrive` at `/content/drive/MyDrive`. In local Jupyter it safely does nothing. Keep the notebook and `run_002_train.npz` in the same Drive folder.

In [ ]:
IN_COLAB = False
DRIVE_ROOT = None
if drive is None:
    print('Not running in Google Colab; Google Drive mount skipped.')
else:
    drive.mount('/content/drive')
    IN_COLAB = True
    DRIVE_ROOT = Path('/content/drive/MyDrive')
    print('Google Drive root:', DRIVE_ROOT)


## 3. Find the adjacent Run 002 file and load EOG

Locally, the notebook checks its own directory and the current working directory. In Google Colab, it mounts Google Drive and searches `MyDrive`. If more than one file named `run_002_train.npz` exists and the notebook directory cannot be determined, set `DATA_DIR` to the intended folder. In Google Colab, it is very likely that you will need to provide the actual drive-based path to the run_002_train.npz file (see comment below for tips).

The raw acquisition stores samples in `data` and the corresponding hardware channel numbers in `channels`. Channel 5 is selected by its hardware ID rather than by assuming a fixed array column.

In [ ]:
DATA_FILENAME = 'run_002_train.npz'
# Optional override, most likely Path('/content/drive/MyDrive/online_lstm/Tutorials')
DATA_DIR = Path('/content/drive/MyDrive/1 - BME PhD/BCI Lab/Superdecoder/online classification/online_lstm/Tutorials')

def find_data_file(filename, data_dir=None):
    """Find a data file beside this notebook locally or anywhere in Google MyDrive."""
    if data_dir is not None:
        path = Path(data_dir).expanduser() / filename
        if not path.is_file():
            raise FileNotFoundError(f'DATA_DIR does not contain {filename}: {path}')
        return path.resolve()

    candidates = []
    try:
        if ipynbname is None:
            raise RuntimeError('ipynbname is unavailable')
        notebook_path = Path(ipynbname.path())
        candidates.append(notebook_path.parent / filename)
    except Exception:
        # Some hosted notebook frontends do not expose their notebook path.
        pass
    candidates.append(Path.cwd() / filename)

    seen = set()
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if candidate.is_file():
            return candidate

    my_drive = globals().get('DRIVE_ROOT')
    if my_drive is not None:
        my_drive = Path(my_drive)
        matches = sorted(path.resolve() for path in my_drive.rglob(filename) if path.is_file())
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            choices = '\n'.join(f'  - {path.parent}' for path in matches)
            raise RuntimeError(
                f'Found multiple Google Drive copies of {filename}:\n{choices}\n'
                'Set DATA_DIR above to the folder containing this notebook and the intended NPZ.'
            )

    raise FileNotFoundError(
        f'Could not find {filename}. Put it beside the notebook. In Colab, store both files '
        'in the same MyDrive folder, or set DATA_DIR explicitly.'
    )


RAW_NPZ = find_data_file(DATA_FILENAME, DATA_DIR)

with np.load(RAW_NPZ, allow_pickle=True) as raw:
    # load_raw_recording/_as_2d_samples_channels converts acquisition data to float32.
    data = np.asarray(raw['data'], dtype=np.float32)
    fs = int(np.asarray(raw['samplerate']).item())
    channels = tuple(int(x) for x in np.asarray(raw['channels']).reshape(-1))

if data.ndim == 1:
    data = data[:, None]
EEG_HARDWARE_CHANNELS = (1, 2, 3, 4)
EEG_CHANNEL_NAMES = ('O1', 'Oz', 'O2', 'POz')
missing_eeg = tuple(channel for channel in EEG_HARDWARE_CHANNELS if channel not in channels)
if missing_eeg:
    raise ValueError(f'EEG channels are absent: {missing_eeg}; acquired channels={channels}')
eeg_raw = data[:, [channels.index(channel) for channel in EEG_HARDWARE_CHANNELS]]

EOG_HARDWARE_CHANNEL = 5
if EOG_HARDWARE_CHANNEL not in channels:
    raise ValueError(f'EOG channel 5 is absent; acquired channels={channels}')
eog_raw = data[:, channels.index(EOG_HARDWARE_CHANNEL)]
time = np.arange(len(eog_raw)) / fs

print('Input:', RAW_NPZ)
print('Sampling rate:', fs, 'Hz')
print('Acquired channels:', channels)
print('EEG shape:', eeg_raw.shape)
print('EOG samples:', len(eog_raw))


## 4. Prepare the EOG detection signal

The pipeline first subtracts the channel median and divides by the median absolute deviation (MAD). It then takes the first difference and applies a zero-phase fourth-order Butterworth low-pass filter at 10 Hz. This is the signal supplied to NeuroKit2.

In [ ]:
# Match preprocessing.normalized_signal_derivative, including its float32 output.
centered = eog_raw - np.nanmedian(eog_raw)
mad = np.nanmedian(np.abs(centered))
scale = mad if np.isfinite(mad) and mad > 0 else 1.0
eog_normalized = centered / scale
eog_derivative = np.diff(eog_normalized, prepend=eog_normalized[:1]).astype(np.float32)

LOWPASS_HZ = 10.0
FILTER_ORDER = 4
sos = butter(FILTER_ORDER, LOWPASS_HZ / (fs / 2), btype='lowpass', output='sos')
try:
    eog_detection_signal = sosfiltfilt(sos, eog_derivative)
except ValueError:
    # This is the production fallback when a signal is too short for filtfilt padding.
    eog_detection_signal = sosfilt(sos, eog_derivative)
eog_detection_signal = eog_detection_signal.astype(np.float32).astype(np.float64)

preview = min(len(eog_raw), 20 * fs)
n_rows = len(EEG_HARDWARE_CHANNELS) + 3
fig, axes = plt.subplots(n_rows, 1, figsize=(14, 1.7 * n_rows), sharex=True)
for index, (name, hardware_channel) in enumerate(zip(EEG_CHANNEL_NAMES, EEG_HARDWARE_CHANNELS)):
    axes[index].plot(time[:preview], eeg_raw[:preview, index], linewidth=.7)
    axes[index].set_ylabel(f'{name} (ch {hardware_channel})')
eog_axis = len(EEG_HARDWARE_CHANNELS)
axes[eog_axis].plot(time[:preview], eog_raw[:preview], color='tab:purple', linewidth=.7)
axes[eog_axis].set_ylabel('Raw EOG (ch 5)')
axes[eog_axis + 1].plot(time[:preview], eog_derivative[:preview], color='tab:gray', linewidth=.7)
axes[eog_axis + 1].set_ylabel('Norm. derivative')
axes[eog_axis + 2].plot(time[:preview], eog_detection_signal[:preview], color='tab:blue', linewidth=.7)
axes[eog_axis + 2].set_ylabel('10 Hz low-pass')
axes[-1].set_xlabel('Time (s)')
for ax in axes:
    ax.grid(alpha=.25)
plt.tight_layout()


## 5. Detect NeuroKit2 events on both polarities

For each polarity, the pipeline calls:

- `nk.eog_clean(..., method='neurokit')`;
- `nk.eog_findpeaks(..., method='brainstorm')`; and
- `nk.eog_features(...)`.

`Blink_LeftZeros` and `Blink_RightZeros` are the event boundaries. Running the detector on the original and inverted trace captures both movement directions. Events shorter than 0.05 s are discarded. The production code also handles a known NeuroKit2 tie-case failure in `eog_features` with the boundary fallback below.

In [ ]:
def fallback_eog_features(cleaned, peaks, fs):
    """Mirror eog_labeling._fallback_eog_features for NeuroKit2's tie edge case."""
    signal = np.asarray(cleaned, dtype=np.float64).reshape(-1)
    peaks = np.asarray(peaks, dtype=np.int64).reshape(-1)
    half_window = max(1, int(round(0.5 * int(fs))))
    leftzeros, rightzeros = [], []
    for peak in peaks:
        peak = int(np.clip(int(peak), 0, max(0, len(signal) - 1)))
        start = max(0, peak - half_window)
        stop = min(len(signal), peak + half_window + 1)
        epoch = signal[start:stop]
        indices = np.arange(start, stop, dtype=np.int64)
        if epoch.size == 0:
            leftzeros.append(peak); rightzeros.append(peak)
            continue
        max_value = float(np.nanmax(epoch))
        max_candidates = indices[np.flatnonzero(epoch == max_value)]
        max_frame = int(max_candidates[0]) if len(max_candidates) else peak
        max_pos = int(np.searchsorted(indices, max_frame))
        signs = np.signbit(epoch)
        crossing_pos = np.flatnonzero(signs[1:] != signs[:-1]) + 1
        crossings = np.sort(np.append(indices[crossing_pos], max_frame))
        max_matches = np.flatnonzero(crossings == max_frame)
        max_crossing_pos = (
            int(max_matches[0]) if len(max_matches)
            else int(np.searchsorted(crossings, max_frame))
        )
        if max_crossing_pos - 1 >= 0:
            leftzero = int(crossings[max_crossing_pos - 1])
        else:
            before = epoch[:max_pos + 1]
            before_indices = indices[:max_pos + 1]
            candidates = before_indices[np.flatnonzero(before == float(np.nanmin(before)))]
            leftzero = int(candidates[-1]) if len(candidates) else max(start, peak - 1)
        if max_crossing_pos + 1 < len(crossings):
            rightzero = int(crossings[max_crossing_pos + 1])
        else:
            after = epoch[max_pos:]
            after_indices = indices[max_pos:]
            candidates = after_indices[np.flatnonzero(after == float(np.nanmin(after)))]
            rightzero = int(candidates[0]) if len(candidates) else min(stop - 1, peak + 1)
        if rightzero <= leftzero:
            leftzero = max(start, peak - 1)
            rightzero = min(stop - 1, peak + 1)
        leftzeros.append(int(leftzero)); rightzeros.append(int(rightzero))
    return pd.DataFrame({
        'Blink_LeftZeros': np.asarray(leftzeros, dtype=np.int64),
        'Blink_RightZeros': np.asarray(rightzeros, dtype=np.int64),
    })


def safe_eog_features(cleaned, peaks, fs):
    try:
        return nk.eog_features(cleaned, peaks, sampling_rate=int(fs))
    except ValueError as exc:
        if 'can only convert an array of size 1 to a Python scalar' not in str(exc):
            raise
        return fallback_eog_features(cleaned, peaks, fs)


MIN_EVENT_DURATION_SEC = 0.05
rows = []

for polarity in (1, -1):
    oriented = polarity * eog_detection_signal
    cleaned = np.asarray(nk.eog_clean(oriented, sampling_rate=fs, method='neurokit'))
    peaks = np.asarray(
        nk.eog_findpeaks(cleaned, sampling_rate=fs, method='brainstorm'),
        dtype=int,
    ).reshape(-1)
    if peaks.size == 0:
        continue
    features = safe_eog_features(cleaned, peaks, fs)
    left_edges = np.asarray(features.get('Blink_LeftZeros', []), dtype=float).reshape(-1)
    right_edges = np.asarray(features.get('Blink_RightZeros', []), dtype=float).reshape(-1)

    for peak, left, right in zip(peaks, left_edges, right_edges):
        if not (np.isfinite(left) and np.isfinite(right)):
            continue
        left = int(np.clip(round(left), 0, len(eog_raw)))
        right = int(np.clip(round(right), 0, len(eog_raw)))
        peak = int(np.clip(peak, 0, len(eog_raw) - 1))
        if right <= left:
            continue
        rows.append({
            'start_sample': left,
            'end_sample': right,
            'peak_sample': peak,
            'polarity': polarity,
            'peak_abs_value': abs(float(cleaned[peak])),
        })

events_unmerged = pd.DataFrame(rows)
if events_unmerged.empty:
    raise RuntimeError('NeuroKit2 did not detect any EOG events.')
min_event_samples = max(1, int(round(MIN_EVENT_DURATION_SEC * fs)))
events_unmerged = events_unmerged[
    (events_unmerged['end_sample'] - events_unmerged['start_sample']) >= min_event_samples
].copy()
if events_unmerged.empty:
    raise RuntimeError('All EOG events were shorter than the minimum duration.')
events_unmerged = events_unmerged.sort_values(
    ['start_sample', 'end_sample', 'peak_abs_value']
).reset_index(drop=True)
print('Events before merging:', len(events_unmerged))
display(events_unmerged.head())


## 6. Merge detections, retain the rightmost end, and select label events

The two polarity passes can detect the same movement more than once. Consecutive events separated by no more than 0.30 s are treated as one event. The merged start is the earliest start and the merged end is the **rightmost (maximum) end**.

The active direct-EOG configuration then ignores endpoints before 6.0 s and requires selected endpoints to be at least 2.0 s apart. There is currently no stop time, peak threshold, or label offset. Each surviving rightmost endpoint becomes a label transition.

In [ ]:
MERGE_GAP_SEC = 0.30
merge_gap_samples = max(0, int(round(MERGE_GAP_SEC * fs)))
merged = []

for event in events_unmerged.to_dict('records'):
    if not merged or event['start_sample'] - merged[-1]['end_sample'] > merge_gap_samples:
        event['merged_neurokit_events'] = 1
        merged.append(event)
        continue

    previous = merged[-1]
    strongest = event if event['peak_abs_value'] > previous['peak_abs_value'] else previous
    merged[-1] = {
        **strongest,
        'start_sample': min(previous['start_sample'], event['start_sample']),
        # This maximum is the rightmost end of the merged EOG event.
        'end_sample': max(previous['end_sample'], event['end_sample']),
        'merged_neurokit_events': previous['merged_neurokit_events'] + 1,
    }

all_merged_events = pd.DataFrame(merged).sort_values(
    ['end_sample', 'start_sample']
).reset_index(drop=True)

# Match detect_eog_label_events and the active direct-EOG configuration.
LABEL_OFFSET_SEC = 0.0
LABEL_EVENT_START_SEC = 6.0
LABEL_EVENT_STOP_SEC = None
LABEL_EVENT_MIN_INTERVAL_SEC = 2.0
LABEL_EVENT_MIN_PEAK_ABS_VALUE = None
label_offset = int(round(LABEL_OFFSET_SEC * fs))
start_sample = int(round(LABEL_EVENT_START_SEC * fs))
stop_sample = None if LABEL_EVENT_STOP_SEC is None else int(round(LABEL_EVENT_STOP_SEC * fs))
min_interval_samples = max(0, int(round(LABEL_EVENT_MIN_INTERVAL_SEC * fs)))
selected = []
last_label_sample = None
for event in all_merged_events.to_dict('records'):
    label_sample = int(np.clip(event['end_sample'] + label_offset, 0, len(eog_raw)))
    if label_sample < start_sample:
        continue
    if stop_sample is not None and label_sample > stop_sample:
        continue
    if (
        LABEL_EVENT_MIN_PEAK_ABS_VALUE is not None
        and event['peak_abs_value'] < LABEL_EVENT_MIN_PEAK_ABS_VALUE
    ):
        continue
    if last_label_sample is not None and label_sample - last_label_sample < min_interval_samples:
        continue
    event['label_transition_sample'] = label_sample
    selected.append(event)
    last_label_sample = label_sample

events = pd.DataFrame(selected).reset_index(drop=True)
if events.empty:
    raise RuntimeError('No EOG events survived the label-event selection rules.')
events.insert(0, 'event_index', np.arange(len(events)))
events['start_time_sec'] = events['start_sample'] / fs
events['rightmost_end_time_sec'] = events['end_sample'] / fs
print('All merged EOG events:', len(all_merged_events))
print('Selected label events:', len(events))
display(events)


## 7. Label at every selected rightmost event end

Start in label 0. At each selected event's `label_transition_sample` (the rightmost end plus the configured zero-second offset), switch to the other binary label. Thus event 0 starts label 1, event 1 returns to label 0, and so on. Samples before the transition retain the previous label; the transition sample and samples after it receive the new label.

In [ ]:
sample_labels = np.zeros(len(eog_raw), dtype=np.int64)
transition_samples = events['label_transition_sample'].to_numpy(dtype=int)

for event_index, transition in enumerate(transition_samples):
    next_transition = (
        transition_samples[event_index + 1]
        if event_index + 1 < len(transition_samples)
        else len(sample_labels)
    )
    new_label = 1 if event_index % 2 == 0 else 0
    sample_labels[transition:next_transition] = new_label

events['label_transition_time_sec'] = transition_samples / fs
events['new_label'] = [1 if i % 2 == 0 else 0 for i in range(len(events))]
display(events[[
    'event_index', 'start_time_sec', 'rightmost_end_time_sec',
    'label_transition_sample', 'new_label', 'merged_neurokit_events',
]])


## 8. Verify event boundaries and labels visually

Every black dashed line is a selected merged event's rightmost endpoint and a label transition. All four raw EEG channels are included for temporal context, followed by raw EOG, the filtered detection signal, and the sample label. Change `START_SEC` and `DURATION_SEC` to inspect the entire recording.

In [ ]:
START_SEC = 0.0
DURATION_SEC = 60.0
lo = max(0, round(START_SEC * fs))
hi = min(len(eog_raw), lo + round(DURATION_SEC * fs))

n_rows = len(EEG_HARDWARE_CHANNELS) + 3
fig, axes = plt.subplots(n_rows, 1, figsize=(15, 1.7 * n_rows), sharex=True)
for index, (name, hardware_channel) in enumerate(zip(EEG_CHANNEL_NAMES, EEG_HARDWARE_CHANNELS)):
    axes[index].plot(time[lo:hi], eeg_raw[lo:hi, index], linewidth=.7, label=f'{name}, channel {hardware_channel}')
    axes[index].set_ylabel(name)
eog_axis = len(EEG_HARDWARE_CHANNELS)
axes[eog_axis].plot(time[lo:hi], eog_raw[lo:hi], color='tab:purple', linewidth=.7, label='raw EOG, channel 5')
axes[eog_axis].set_ylabel('Raw EOG')
axes[eog_axis + 1].plot(time[lo:hi], eog_detection_signal[lo:hi], color='tab:blue', linewidth=.7, label='filtered detection signal')
axes[eog_axis + 1].set_ylabel('EOG detect')
axes[-1].step(time[lo:hi], sample_labels[lo:hi], where='post', color='black', label='sample label')
for row in events.itertuples():
    if row.end_sample < lo or row.start_sample > hi:
        continue
    axes[eog_axis].axvspan(row.start_sample / fs, row.end_sample / fs, color='tab:green', alpha=.18)
    axes[eog_axis + 1].axvspan(row.start_sample / fs, row.end_sample / fs, color='tab:green', alpha=.18)
    for ax in axes:
        ax.axvline(row.end_sample / fs, color='black', linestyle='--', alpha=.7)
axes[-1].set_ylabel('Label')
axes[-1].set_yticks([0, 1])
axes[-1].set_xlabel('Recording time (s)')
for ax in axes:
    ax.legend(loc='upper right')
    ax.grid(alpha=.25)
plt.tight_layout()


## Key rule

NeuroKit2 supplies a left and right boundary for each detection. After both-polarity detections are merged, use `max(end_sample)` as the merged event's rightmost edge. Sort the edges chronologically, apply the configured 6 s start and 2 s minimum-interval selection rules, and alternate the binary label at every surviving edge. No audio timing is involved.